# 30_PF008_PF011 - Módulos productivos de inferencia

Objetivo: convertir la lógica experimental de inferencia y análisis visual en módulos reutilizables dentro de `pfi_ai_service`.

Tickets cubiertos:

- PF-008: convertir pipeline E13 a módulos productivos.
- PF-009: soportar entrada de archivo.
- PF-010: generar overlays finales.
- PF-011: calcular mediciones geométricas mínimas.

Este notebook no entrena modelos ni expone endpoints finales. Prepara el paquete Python para que el bloque siguiente pueda agregar endpoints reales.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json, os, sys, textwrap, shutil, hashlib, datetime
import pandas as pd
import numpy as np

PFI_ROOT = Path("/content/drive/MyDrive/PFI_MVP")
REPO_ROOT = PFI_ROOT / "repo"
SERVICE_ROOT = REPO_ROOT / "ai_service"
PACKAGE_ROOT = SERVICE_ROOT / "pfi_ai_service"

RESULTS_ROOT = PFI_ROOT / "results" / "PF008_PF011_product_inference_modules"
DOCS_ROOT = REPO_ROOT / "docs"
BACKLOG_ROOT = REPO_ROOT / "backlogProducto"

for p in [PACKAGE_ROOT, RESULTS_ROOT, DOCS_ROOT, BACKLOG_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

print("PFI_ROOT:", PFI_ROOT)
print("REPO_ROOT:", REPO_ROOT, "| exists:", REPO_ROOT.exists())
print("PACKAGE_ROOT:", PACKAGE_ROOT, "| exists:", PACKAGE_ROOT.exists())
print("RESULTS_ROOT:", RESULTS_ROOT)

## 1. Validación de entradas del bloque anterior

In [ ]:
DATA_FREEZE_CONFIG = REPO_ROOT / "config" / "data_freeze_config.json"
MODEL_REGISTRY_FINAL = REPO_ROOT / "config" / "model_registry_final.json"

print("DATA_FREEZE_CONFIG:", DATA_FREEZE_CONFIG, "exists:", DATA_FREEZE_CONFIG.exists())
print("MODEL_REGISTRY_FINAL:", MODEL_REGISTRY_FINAL, "exists:", MODEL_REGISTRY_FINAL.exists())

if not DATA_FREEZE_CONFIG.exists():
    raise FileNotFoundError("No existe config/data_freeze_config.json. Correr PF001-PF003 primero.")
if not MODEL_REGISTRY_FINAL.exists():
    raise FileNotFoundError("No existe config/model_registry_final.json. Correr PF004-PF007 primero.")

data_freeze = json.loads(DATA_FREEZE_CONFIG.read_text(encoding="utf-8"))
model_registry = json.loads(MODEL_REGISTRY_FINAL.read_text(encoding="utf-8"))

print("Registry keys:", list(model_registry.get("models", model_registry).keys()) if isinstance(model_registry, dict) else type(model_registry))

## 2. Escritura de módulos productivos

In [ ]:
modules = {
    "preprocessing.py": "\nfrom __future__ import annotations\n\nfrom pathlib import Path\nfrom typing import Any, Dict, Optional, Tuple\n\nimport numpy as np\n\n\ndef normalize_minmax(image: np.ndarray, eps: float = 1e-8) -> np.ndarray:\n    \"\"\"Normaliza una imagen a rango [0, 1].\"\"\"\n    arr = np.asarray(image, dtype=np.float32)\n    finite = np.isfinite(arr)\n    if not finite.any():\n        return np.zeros_like(arr, dtype=np.float32)\n    lo = float(np.nanmin(arr[finite]))\n    hi = float(np.nanmax(arr[finite]))\n    if abs(hi - lo) < eps:\n        return np.zeros_like(arr, dtype=np.float32)\n    return ((arr - lo) / (hi - lo + eps)).astype(np.float32)\n\n\ndef zscore_clip(image: np.ndarray, clip_percentiles: Tuple[float, float] = (1, 99), eps: float = 1e-8) -> np.ndarray:\n    \"\"\"Normaliza con percentiles y z-score robusto, \u00fatil para RM.\"\"\"\n    arr = np.asarray(image, dtype=np.float32)\n    finite = np.isfinite(arr)\n    if not finite.any():\n        return np.zeros_like(arr, dtype=np.float32)\n    lo, hi = np.percentile(arr[finite], clip_percentiles)\n    arr = np.clip(arr, lo, hi)\n    mean = float(arr[finite].mean())\n    std = float(arr[finite].std())\n    if std < eps:\n        return np.zeros_like(arr, dtype=np.float32)\n    return ((arr - mean) / (std + eps)).astype(np.float32)\n\n\ndef ensure_2d(image: np.ndarray) -> np.ndarray:\n    \"\"\"Convierte un array a imagen 2D tomando un corte central si hace falta.\"\"\"\n    arr = np.asarray(image)\n    if arr.ndim == 2:\n        return arr\n    if arr.ndim == 3:\n        axis = int(np.argmin(arr.shape))\n        idx = arr.shape[axis] // 2\n        return np.take(arr, idx, axis=axis)\n    raise ValueError(f\"Se esperaba imagen 2D o 3D; recibido shape={arr.shape}\")\n\n\ndef load_image_file(path: str | Path) -> Dict[str, Any]:\n    \"\"\"Carga imagen desde formatos simples usados en el MVP.\n\n    Soporta:\n    - .npy\n    - .png/.jpg/.jpeg v\u00eda PIL\n    - .mha/.mhd si SimpleITK est\u00e1 instalado\n\n    Devuelve un dict con image, path, suffix, spacing y metadata.\n    \"\"\"\n    path = Path(path)\n    if not path.exists():\n        raise FileNotFoundError(path)\n\n    suffix = path.suffix.lower()\n    spacing = None\n    metadata: Dict[str, Any] = {}\n\n    if suffix == \".npy\":\n        image = np.load(path)\n    elif suffix in {\".png\", \".jpg\", \".jpeg\"}:\n        from PIL import Image\n        image = np.array(Image.open(path).convert(\"L\"))\n    elif suffix in {\".mha\", \".mhd\"}:\n        try:\n            import SimpleITK as sitk\n        except Exception as exc:\n            raise ImportError(\"Para cargar .mha/.mhd instalar SimpleITK.\") from exc\n        itk_img = sitk.ReadImage(str(path))\n        image = sitk.GetArrayFromImage(itk_img)\n        spacing = tuple(float(x) for x in itk_img.GetSpacing())\n        metadata[\"origin\"] = tuple(float(x) for x in itk_img.GetOrigin())\n        metadata[\"direction\"] = tuple(float(x) for x in itk_img.GetDirection())\n    else:\n        raise ValueError(f\"Formato no soportado para carga directa: {suffix}\")\n\n    return {\n        \"image\": image,\n        \"path\": str(path),\n        \"suffix\": suffix,\n        \"spacing\": spacing,\n        \"metadata\": metadata,\n    }\n\n\ndef resize_nearest(mask: np.ndarray, size: Tuple[int, int]) -> np.ndarray:\n    \"\"\"Resize de m\u00e1scara con vecino m\u00e1s cercano.\"\"\"\n    from PIL import Image\n    arr = np.asarray(mask)\n    im = Image.fromarray(arr.astype(np.uint8))\n    return np.array(im.resize((size[1], size[0]), resample=Image.Resampling.NEAREST))\n\n\ndef resize_image_bilinear(image: np.ndarray, size: Tuple[int, int]) -> np.ndarray:\n    \"\"\"Resize de imagen con bilineal.\"\"\"\n    from PIL import Image\n    arr = normalize_minmax(image)\n    im = Image.fromarray((arr * 255).astype(np.uint8))\n    out = np.array(im.resize((size[1], size[0]), resample=Image.Resampling.BILINEAR)).astype(np.float32) / 255.0\n    return out\n",
    "measurements.py": "\nfrom __future__ import annotations\n\nfrom typing import Any, Dict, List, Optional, Tuple\n\nimport numpy as np\n\n\ndef foreground_ratio(mask: np.ndarray, background_label: int = 0) -> float:\n    arr = np.asarray(mask)\n    if arr.size == 0:\n        return 0.0\n    return float(np.mean(arr != background_label))\n\n\ndef present_classes(mask: np.ndarray, background_label: int = 0) -> List[int]:\n    values = np.unique(np.asarray(mask))\n    return [int(v) for v in values if int(v) != background_label]\n\n\ndef class_pixel_counts(mask: np.ndarray) -> Dict[int, int]:\n    values, counts = np.unique(np.asarray(mask), return_counts=True)\n    return {int(v): int(c) for v, c in zip(values, counts)}\n\n\ndef bounding_box_2d(binary_mask: np.ndarray) -> Optional[Tuple[int, int, int, int]]:\n    ys, xs = np.where(np.asarray(binary_mask).astype(bool))\n    if len(xs) == 0:\n        return None\n    return int(ys.min()), int(xs.min()), int(ys.max()), int(xs.max())\n\n\ndef connected_components_count(binary_mask: np.ndarray) -> int:\n    \"\"\"Cuenta componentes conexas 2D con scipy si est\u00e1 disponible; si no, BFS simple.\"\"\"\n    arr = np.asarray(binary_mask).astype(bool)\n    if arr.ndim != 2:\n        arr = np.squeeze(arr)\n    if arr.ndim != 2:\n        raise ValueError(f\"connected_components_count espera 2D, recibido {arr.shape}\")\n\n    try:\n        from scipy import ndimage as ndi\n        _, n = ndi.label(arr)\n        return int(n)\n    except Exception:\n        visited = np.zeros(arr.shape, dtype=bool)\n        h, w = arr.shape\n        n = 0\n        for y in range(h):\n            for x in range(w):\n                if arr[y, x] and not visited[y, x]:\n                    n += 1\n                    stack = [(y, x)]\n                    visited[y, x] = True\n                    while stack:\n                        cy, cx = stack.pop()\n                        for dy, dx in ((1,0),(-1,0),(0,1),(0,-1)):\n                            ny, nx = cy + dy, cx + dx\n                            if 0 <= ny < h and 0 <= nx < w and arr[ny, nx] and not visited[ny, nx]:\n                                visited[ny, nx] = True\n                                stack.append((ny, nx))\n        return int(n)\n\n\ndef mask_measurements(mask: np.ndarray, spacing: Optional[Tuple[float, ...]] = None, background_label: int = 0) -> Dict[str, Any]:\n    \"\"\"Calcula mediciones geom\u00e9tricas m\u00ednimas por clase.\n\n    Las mediciones son descriptivas y no diagn\u00f3sticas.\n    \"\"\"\n    arr = np.asarray(mask)\n    if arr.ndim != 2:\n        arr = np.squeeze(arr)\n    if arr.ndim != 2:\n        raise ValueError(f\"mask_measurements espera m\u00e1scara 2D, recibido {arr.shape}\")\n\n    pixel_area = None\n    if spacing is not None and len(spacing) >= 2:\n        pixel_area = float(spacing[0]) * float(spacing[1])\n\n    rows = []\n    for cls, count in class_pixel_counts(arr).items():\n        if cls == background_label:\n            continue\n        binary = arr == cls\n        bbox = bounding_box_2d(binary)\n        ncomp = connected_components_count(binary)\n        row = {\n            \"class_id\": int(cls),\n            \"pixel_count\": int(count),\n            \"foreground_ratio_class\": float(count / arr.size),\n            \"n_components\": int(ncomp),\n            \"bbox_ymin_xmin_ymax_xmax\": bbox,\n        }\n        if pixel_area is not None:\n            row[\"area_mm2_approx\"] = float(count * pixel_area)\n        rows.append(row)\n\n    return {\n        \"shape\": tuple(int(x) for x in arr.shape),\n        \"foreground_ratio\": foreground_ratio(arr, background_label=background_label),\n        \"present_classes\": present_classes(arr, background_label=background_label),\n        \"n_components_foreground\": connected_components_count(arr != background_label),\n        \"class_measurements\": rows,\n        \"notes\": \"Mediciones geom\u00e9tricas descriptivas; no constituyen diagn\u00f3stico cl\u00ednico.\",\n    }\n",
    "overlay.py": "\nfrom __future__ import annotations\n\nfrom pathlib import Path\nfrom typing import Dict, Optional, Tuple\n\nimport numpy as np\n\n\nDEFAULT_PALETTE = {\n    0: (0, 0, 0),\n    1: (230, 25, 75),\n    2: (60, 180, 75),\n    3: (0, 130, 200),\n    4: (245, 130, 48),\n    5: (145, 30, 180),\n    6: (70, 240, 240),\n}\n\n\ndef to_uint8_grayscale(image: np.ndarray) -> np.ndarray:\n    arr = np.asarray(image, dtype=np.float32)\n    finite = np.isfinite(arr)\n    if not finite.any():\n        return np.zeros(arr.shape, dtype=np.uint8)\n    lo = float(np.nanmin(arr[finite]))\n    hi = float(np.nanmax(arr[finite]))\n    if abs(hi - lo) < 1e-8:\n        return np.zeros(arr.shape, dtype=np.uint8)\n    arr = (arr - lo) / (hi - lo + 1e-8)\n    return np.clip(arr * 255, 0, 255).astype(np.uint8)\n\n\ndef make_overlay_rgb(image: np.ndarray, mask: np.ndarray, alpha: float = 0.35, palette: Optional[Dict[int, Tuple[int, int, int]]] = None) -> np.ndarray:\n    \"\"\"Genera overlay RGB a partir de imagen 2D y m\u00e1scara multiclase.\"\"\"\n    palette = palette or DEFAULT_PALETTE\n    img = to_uint8_grayscale(image)\n    if img.ndim != 2:\n        img = np.squeeze(img)\n    mask_arr = np.asarray(mask)\n    if mask_arr.ndim != 2:\n        mask_arr = np.squeeze(mask_arr)\n    if img.shape != mask_arr.shape:\n        raise ValueError(f\"image y mask deben tener mismo shape; {img.shape} != {mask_arr.shape}\")\n\n    rgb = np.stack([img, img, img], axis=-1).astype(np.float32)\n    for cls in np.unique(mask_arr):\n        cls = int(cls)\n        if cls == 0:\n            continue\n        color = np.array(palette.get(cls, (255, 255, 0)), dtype=np.float32)\n        idx = mask_arr == cls\n        rgb[idx] = (1 - alpha) * rgb[idx] + alpha * color\n    return np.clip(rgb, 0, 255).astype(np.uint8)\n\n\ndef save_overlay_png(path: str | Path, image: np.ndarray, mask: np.ndarray, alpha: float = 0.35) -> Path:\n    from PIL import Image\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    overlay = make_overlay_rgb(image, mask, alpha=alpha)\n    Image.fromarray(overlay).save(path)\n    return path\n",
    "inference.py": "\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom pathlib import Path\nfrom typing import Any, Dict, Optional\n\nimport json\nimport numpy as np\n\n\n@dataclass\nclass ModelArtifact:\n    model_key: str\n    plane: str\n    dataset_key: str\n    checkpoint_path: Path\n    num_classes: int\n    target_size: tuple[int, int]\n    class_names: Dict[str, str]\n    sha256: Optional[str] = None\n\n\ndef _models_dict(registry: Dict[str, Any]) -> Dict[str, Any]:\n    if \"models\" in registry and isinstance(registry[\"models\"], dict):\n        return registry[\"models\"]\n    return registry\n\n\ndef load_model_registry(path: str | Path) -> Dict[str, Any]:\n    path = Path(path)\n    if not path.exists():\n        raise FileNotFoundError(path)\n    return json.loads(path.read_text(encoding=\"utf-8\"))\n\n\ndef get_model_artifact(registry: Dict[str, Any], model_key: str) -> ModelArtifact:\n    models = _models_dict(registry)\n    if model_key not in models:\n        raise KeyError(f\"Modelo no encontrado en registry: {model_key}\")\n\n    item = models[model_key]\n    checkpoint = item.get(\"final_path\") or item.get(\"checkpoint_path\") or item.get(\"model_path\")\n    if checkpoint is None:\n        raise KeyError(f\"El modelo {model_key} no tiene ruta de checkpoint en el registry.\")\n\n    target_size = item.get(\"target_size\") or item.get(\"input_size\") or [256, 256]\n    class_names = item.get(\"class_names\") or {}\n\n    return ModelArtifact(\n        model_key=model_key,\n        plane=item.get(\"plane\", \"unknown\"),\n        dataset_key=item.get(\"dataset_key\", \"unknown\"),\n        checkpoint_path=Path(checkpoint),\n        num_classes=int(item.get(\"num_classes\", len(class_names) if class_names else 0)),\n        target_size=(int(target_size[0]), int(target_size[1])),\n        class_names={str(k): str(v) for k, v in class_names.items()},\n        sha256=item.get(\"sha256\"),\n    )\n\n\nclass ProductInferenceEngine:\n    \"\"\"Interfaz productiva para inferencia.\n\n    Esta clase deja un contrato estable para PF-012/PF-019.\n    El m\u00e9todo `predict_mask` admite dos modos:\n    - si se pasa `mock_mask`, devuelve esa m\u00e1scara para smoke tests;\n    - si se implementa un predictor real, debe reemplazar `_predict_with_torch`.\n    \"\"\"\n\n    def __init__(self, registry_path: str | Path):\n        self.registry_path = Path(registry_path)\n        self.registry = load_model_registry(self.registry_path)\n\n    def artifact(self, model_key: str) -> ModelArtifact:\n        return get_model_artifact(self.registry, model_key)\n\n    def validate_artifact(self, model_key: str) -> Dict[str, Any]:\n        art = self.artifact(model_key)\n        return {\n            \"model_key\": art.model_key,\n            \"plane\": art.plane,\n            \"dataset_key\": art.dataset_key,\n            \"checkpoint_path\": str(art.checkpoint_path),\n            \"checkpoint_exists\": art.checkpoint_path.exists(),\n            \"num_classes\": art.num_classes,\n            \"target_size\": art.target_size,\n            \"human_review_required\": True,\n        }\n\n    def predict_mask(self, image: np.ndarray, model_key: str, mock_mask: Optional[np.ndarray] = None) -> Dict[str, Any]:\n        art = self.artifact(model_key)\n        if mock_mask is not None:\n            pred = np.asarray(mock_mask).astype(np.uint8)\n            return {\n                \"model_key\": model_key,\n                \"plane\": art.plane,\n                \"prediction_mask\": pred,\n                \"mode\": \"mock_mask_smoke_test\",\n                \"human_review_required\": True,\n            }\n        raise NotImplementedError(\n            \"La inferencia torch real se conectar\u00e1 en el pr\u00f3ximo bloque. \"\n            \"PF-008/PF-011 define contrato, carga de registry y m\u00f3dulos reutilizables.\"\n        )\n",
    "pipeline.py": "\nfrom __future__ import annotations\n\nfrom pathlib import Path\nfrom typing import Any, Dict, Optional\n\nimport numpy as np\n\nfrom .preprocessing import ensure_2d, load_image_file, normalize_minmax\nfrom .measurements import mask_measurements\nfrom .overlay import save_overlay_png\nfrom .inference import ProductInferenceEngine\n\n\ndef run_case_from_arrays(\n    *,\n    image: np.ndarray,\n    mask: np.ndarray,\n    model_key: str,\n    output_dir: str | Path,\n    case_id: str = \"case_smoke\",\n    spacing: Optional[tuple[float, ...]] = None,\n) -> Dict[str, Any]:\n    \"\"\"Ejecuta pipeline productivo m\u00ednimo sobre arrays ya cargados.\n\n    Este flujo permite testear preprocesamiento, mediciones y overlay sin depender todav\u00eda\n    del endpoint ni del predictor torch real.\n    \"\"\"\n    output_dir = Path(output_dir)\n    output_dir.mkdir(parents=True, exist_ok=True)\n\n    image2d = ensure_2d(image)\n    mask2d = ensure_2d(mask).astype(np.uint8)\n    image_norm = normalize_minmax(image2d)\n\n    overlay_path = output_dir / f\"{case_id}_{model_key}_overlay.png\"\n    save_overlay_png(overlay_path, image_norm, mask2d)\n\n    measurements = mask_measurements(mask2d, spacing=spacing)\n\n    return {\n        \"case_id\": case_id,\n        \"model_key\": model_key,\n        \"image_shape\": tuple(int(x) for x in image2d.shape),\n        \"mask_shape\": tuple(int(x) for x in mask2d.shape),\n        \"overlay_path\": str(overlay_path),\n        \"measurements\": measurements,\n        \"human_review_required\": True,\n    }\n\n\ndef run_case_from_file_with_mock_mask(\n    *,\n    image_path: str | Path,\n    mask: np.ndarray,\n    model_key: str,\n    output_dir: str | Path,\n    case_id: str = \"case_file_smoke\",\n) -> Dict[str, Any]:\n    loaded = load_image_file(image_path)\n    return run_case_from_arrays(\n        image=loaded[\"image\"],\n        mask=mask,\n        model_key=model_key,\n        output_dir=output_dir,\n        case_id=case_id,\n        spacing=loaded.get(\"spacing\"),\n    )\n\n\ndef validate_product_inference_setup(registry_path: str | Path, model_keys: list[str]) -> Dict[str, Any]:\n    engine = ProductInferenceEngine(registry_path)\n    artifacts = [engine.validate_artifact(k) for k in model_keys]\n    return {\n        \"registry_path\": str(registry_path),\n        \"models\": artifacts,\n        \"all_checkpoints_exist\": bool(all(x[\"checkpoint_exists\"] for x in artifacts)),\n        \"human_review_required\": True,\n    }\n",
}

for name, content in modules.items():
    path = PACKAGE_ROOT / name
    path.write_text(content.strip() + "\n", encoding="utf-8")
    print("Wrote:", path)

## 3. Inventario de módulos

In [ ]:
module_rows = []
for name in ["preprocessing.py", "measurements.py", "overlay.py", "inference.py", "pipeline.py"]:
    path = PACKAGE_ROOT / name
    module_rows.append({
        "relative_path": str(path.relative_to(REPO_ROOT)),
        "exists": path.exists(),
        "size_bytes": path.stat().st_size if path.exists() else 0,
    })

module_inventory_df = pd.DataFrame(module_rows)
module_inventory_csv = RESULTS_ROOT / "PF008_PF011_module_inventory.csv"
module_inventory_df.to_csv(module_inventory_csv, index=False)
display(module_inventory_df)
print("Module inventory:", module_inventory_csv)

## 4. Smoke test de módulos

In [ ]:
# Importación limpia del paquete
if str(SERVICE_ROOT) not in sys.path:
    sys.path.insert(0, str(SERVICE_ROOT))

# Forzar reload de módulos
for mod in list(sys.modules.keys()):
    if mod.startswith("pfi_ai_service."):
        del sys.modules[mod]

from pfi_ai_service.preprocessing import normalize_minmax, ensure_2d
from pfi_ai_service.measurements import mask_measurements, foreground_ratio, present_classes
from pfi_ai_service.overlay import save_overlay_png
from pfi_ai_service.inference import ProductInferenceEngine
from pfi_ai_service.pipeline import run_case_from_arrays, validate_product_inference_setup

# Synthetic smoke input
rng = np.random.default_rng(42)
image = rng.normal(loc=100, scale=25, size=(128, 128)).astype(np.float32)
mask = np.zeros((128, 128), dtype=np.uint8)
mask[25:75, 35:85] = 1
mask[70:110, 60:115] = 2
mask[45:95, 15:28] = 3

model_keys = ["axial_t2_alkafri", "sagittal_spider"]
setup_report = validate_product_inference_setup(MODEL_REGISTRY_FINAL, model_keys)

case_report = run_case_from_arrays(
    image=image,
    mask=mask,
    model_key="sagittal_spider",
    output_dir=RESULTS_ROOT / "smoke_overlays",
    case_id="synthetic_smoke",
    spacing=(0.7, 0.7),
)

smoke_report = {
    "module_import_ok": True,
    "setup_report": setup_report,
    "case_report": case_report,
    "decision": "PF008_PF011_modules_import_and_smoke_test_passed",
}

smoke_path = RESULTS_ROOT / "PF008_PF011_smoke_test_report.json"
smoke_path.write_text(json.dumps(smoke_report, indent=2, ensure_ascii=False), encoding="utf-8")

print("Smoke report:", smoke_path)
print(json.dumps({
    "module_import_ok": smoke_report["module_import_ok"],
    "all_checkpoints_exist": setup_report["all_checkpoints_exist"],
    "overlay_path": case_report["overlay_path"],
    "foreground_ratio": case_report["measurements"]["foreground_ratio"],
    "present_classes": case_report["measurements"]["present_classes"],
    "decision": smoke_report["decision"],
}, indent=2, ensure_ascii=False))

## 5. Checks finales

In [ ]:
checks = [
    ("model_registry_final_exists", MODEL_REGISTRY_FINAL.exists(), str(MODEL_REGISTRY_FINAL)),
    ("preprocessing_module_written", (PACKAGE_ROOT / "preprocessing.py").exists(), str(PACKAGE_ROOT / "preprocessing.py")),
    ("measurements_module_written", (PACKAGE_ROOT / "measurements.py").exists(), str(PACKAGE_ROOT / "measurements.py")),
    ("overlay_module_written", (PACKAGE_ROOT / "overlay.py").exists(), str(PACKAGE_ROOT / "overlay.py")),
    ("inference_module_written", (PACKAGE_ROOT / "inference.py").exists(), str(PACKAGE_ROOT / "inference.py")),
    ("pipeline_module_written", (PACKAGE_ROOT / "pipeline.py").exists(), str(PACKAGE_ROOT / "pipeline.py")),
    ("module_inventory_written", module_inventory_csv.exists(), str(module_inventory_csv)),
    ("smoke_report_written", smoke_path.exists(), str(smoke_path)),
    ("all_final_checkpoints_exist", setup_report["all_checkpoints_exist"], str(MODEL_REGISTRY_FINAL)),
    ("overlay_png_written", Path(case_report["overlay_path"]).exists(), case_report["overlay_path"]),
    ("measurements_have_classes", len(case_report["measurements"]["present_classes"]) > 0, str(case_report["measurements"]["present_classes"])),
]

checks_df = pd.DataFrame(checks, columns=["check", "ok", "detail"])
checks_csv = RESULTS_ROOT / "PF008_PF011_checks.csv"
checks_df.to_csv(checks_csv, index=False)
display(checks_df)
print("Checks:", checks_csv)
print("All OK:", bool(checks_df["ok"].all()))

## 6. Documentación y reporte

In [ ]:
docs_md = f"""# PF-008 a PF-011 - Módulos productivos de inferencia

## Objetivo

Convertir la lógica experimental de inferencia y visualización en módulos reutilizables dentro del paquete `pfi_ai_service`.

## Módulos generados

- `preprocessing.py`: carga básica de archivos, normalización y preparación 2D.
- `measurements.py`: foreground ratio, clases presentes, componentes y mediciones geométricas mínimas.
- `overlay.py`: generación de overlay RGB sobre imagen original.
- `inference.py`: carga del registry final e interfaz estable de motor de inferencia.
- `pipeline.py`: flujo mínimo de caso para arrays o archivo con máscara mock/smoke.

## Alcance actual

Este bloque no expone endpoints FastAPI finales ni ejecuta todavía inferencia torch real desde archivo. Deja contratos y módulos importables para el próximo bloque.

## Smoke test

- módulos importados correctamente;
- registry final cargado;
- checkpoints finales encontrados;
- overlay sintético generado;
- mediciones mínimas calculadas;
- human_review_required mantenido.

## Decisión

`PF008_PF011_modules_ready_for_service_endpoints`

## Política metodológica

El sistema continúa siendo de apoyo. Las mediciones y overlays son salidas revisables y no constituyen diagnóstico clínico.
"""

docs_path = DOCS_ROOT / "PF008_PF011_product_inference_modules.md"
docs_path.write_text(docs_md, encoding="utf-8")

report = {
    "notebook": "30_PF008_PF011_product_inference_modules",
    "goal": "convert E13 inference logic into reusable product inference modules",
    "timestamp": datetime.datetime.now().isoformat(timespec="seconds"),
    "inputs": {
        "model_registry_final": str(MODEL_REGISTRY_FINAL),
        "data_freeze_config": str(DATA_FREEZE_CONFIG),
    },
    "outputs": {
        "module_inventory_csv": str(module_inventory_csv),
        "smoke_test_report_json": str(smoke_path),
        "checks_csv": str(checks_csv),
        "docs_repo": str(docs_path),
        "package_root": str(PACKAGE_ROOT),
    },
    "summary": {
        "modules_written": int(len(module_inventory_df)),
        "all_checks_ok": bool(checks_df["ok"].all()),
        "model_keys": model_keys,
        "all_final_checkpoints_exist": bool(setup_report["all_checkpoints_exist"]),
        "smoke_overlay_path": case_report["overlay_path"],
        "smoke_present_classes": case_report["measurements"]["present_classes"],
    },
    "methodological_policy": {
        "human_review_required": True,
        "not_clinical_diagnosis": True,
        "not_real_3d_reconstruction": True,
        "product_scope": "MVP técnico con inferencia 2D multiplanar modular",
    },
    "decision": "PF008_PF011_modules_ready_for_service_endpoints",
}
report_path = RESULTS_ROOT / "PF008_PF011_report.json"
report_path.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")

closure_md = f"""# Cierre PF-008 a PF-011 - Módulos productivos de inferencia

Estado: cerrado si todos los checks son correctos.

## Resultado

Se generaron módulos productivos de inferencia en el paquete `pfi_ai_service`.

## Módulos

- preprocessing.py
- measurements.py
- overlay.py
- inference.py
- pipeline.py

## Decisión

`PF008_PF011_modules_ready_for_service_endpoints`

## Próximo bloque

PF-012 a PF-019: endurecer agente y exponer endpoints reales del servicio Python.
"""

closure_path = BACKLOG_ROOT / "PF008_PF011_resultados_cierre.md"
closure_path.write_text(closure_md, encoding="utf-8")

print("Docs:", docs_path)
print("Report:", report_path)
print("Closure draft:", closure_path)
print(json.dumps(report, indent=2, ensure_ascii=False))

## 7. Comandos sugeridos para commit

In [ ]:
print('''%cd /content/drive/MyDrive/PFI_MVP/repo
!git status
!git add ai_service/pfi_ai_service/preprocessing.py ai_service/pfi_ai_service/measurements.py ai_service/pfi_ai_service/overlay.py ai_service/pfi_ai_service/inference.py ai_service/pfi_ai_service/pipeline.py docs/PF008_PF011_product_inference_modules.md backlogProducto/PF008_PF011_resultados_cierre.md notebooks/30_PF008_PF011_product_inference_modules.ipynb
!git commit -m "Add PF008 PF011 product inference modules"
!git push''')